# 08 - Build the dashboard

**Goal:** assemble the outputs of notebooks 02-07 into one interactive, self-contained dashboard.

**Inputs:** `data/processed/` outputs and `dashboard/template.html`
**Outputs:**
- `dashboard/retail_dashboard.html`: open it in any browser (no server, no internet needed). It has five tabs (Overview, Retention, Segments, Churn risk, Actions), hover tooltips, a table view on every chart and a light/dark theme switch.
- `dashboard/tableau_data/*.csv`: tidy, Tableau-ready tables (real dates, long formats, sort keys) for building the dashboard in Tableau; see `dashboard/tableau_build_guide.md`.

The template holds all layout and chart code; this notebook only prepares the numbers, so every figure on the dashboard traces back to a notebook output.

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent  # notebook runs from notebooks/
PROCESSED_DIR = ROOT / "data" / "processed"
DASH_DIR = ROOT / "dashboard"

def records(frame):
    """DataFrame -> list of plain dicts with NaN as None, ready for JSON."""
    return json.loads(frame.to_json(orient="records"))

## 1. Load

In [2]:
df = pd.read_csv(
    PROCESSED_DIR / "transactions_clean.csv",
    parse_dates=["invoice_date", "invoice_month", "first_purchase_month"],
    dtype={"invoice": str, "stock_code": str, "customer_id": "Int64"},
)
sales = df[~df.is_cancellation]
cancels = df[df.is_cancellation]
tracked = sales[sales.has_customer]

rfm_seg = pd.read_csv(PROCESSED_DIR / "rfm_segments.csv")
rfm_cust = pd.read_csv(PROCESSED_DIR / "rfm_customers.csv")
cohort = pd.read_csv(PROCESSED_DIR / "cohort_retention.csv", parse_dates=["cohort"])
repeat90 = pd.read_csv(PROCESSED_DIR / "repeat_rate_90d.csv")
new_cust = pd.read_csv(PROCESSED_DIR / "new_customers_by_month.csv", parse_dates=["cohort"])
bands = pd.read_csv(PROCESSED_DIR / "churn_by_recency.csv")
backtest = pd.read_csv(PROCESSED_DIR / "churn_rule_backtest.csv")
risk = pd.read_csv(PROCESSED_DIR / "at_risk_customers.csv")
recs = pd.read_csv(PROCESSED_DIR / "recommendations.csv")
print("loaded all inputs")

loaded all inputs


## 2. Monthly table

In [3]:
LAST_FULL = pd.Timestamp("2011-11-01")

monthly = pd.DataFrame({
    "gross": sales.groupby("invoice_month").line_total.sum(),
    "orders": sales.groupby("invoice_month").invoice.nunique(),
    "customers": tracked.groupby("invoice_month").customer_id.nunique(),
    "cancelled": -cancels.groupby("invoice_month").line_total.sum(),
})
is_new = tracked.invoice_month == tracked.first_purchase_month
monthly["new"] = tracked[is_new].groupby("invoice_month").customer_id.nunique()
monthly = monthly.fillna(0)
monthly["returning"] = monthly.customers - monthly.new
monthly["aov"] = monthly.gross / monthly.orders
monthly["partial"] = monthly.index > LAST_FULL
monthly = monthly.reset_index().assign(m=lambda d: d.invoice_month.dt.strftime("%Y-%m")).drop(columns="invoice_month")
monthly.round(2).tail(3)

,gross,orders,customers,cancelled,new,returning,aov,partial,m
22,1151263.73,2040,1364,81895.50,221,1143,564.34,False,2011-10
23,1503866.78,2769,1664,47720.98,191,1473,543.11,False,2011-11
24,637808.33,819,615,205089.27,28,587,778.76,True,2011-12


## 3. Headline numbers

In [4]:
# Pareto curve on tracked gross sales, 0-100% of customers
cust_sales = tracked.groupby("customer_id").line_total.sum().sort_values(ascending=False)
cum = cust_sales.cumsum() / cust_sales.sum()
pareto = [[0, 0.0]] + [[p, round(float(cum.iloc[max(int(len(cum) * p / 100) - 1, 0)]) * 100, 2)] for p in range(1, 101)]

orders_per_customer = tracked.groupby("customer_id").invoice.nunique()
jan_nov = sales[sales.invoice_date.dt.month <= 11].groupby(sales.invoice_date.dt.year).line_total.sum()

jun_nov = new_cust[new_cust.cohort.dt.month.between(6, 11) & new_cust.cohort.dt.year.isin([2010, 2011])]
jun_nov_totals = jun_nov.assign(total=jun_nov.UK + jun_nov["Non-UK"]).groupby(jun_nov.cohort.dt.year).total.sum()

flagged = risk[risk.flag_90d & (risk.trailing_rev > 0)]
precision = float(backtest.loc[backtest.rule == "silent > 90 days", "precision"].iloc[0])
seg = rfm_seg.set_index("segment")
quiet = rfm_cust[(rfm_cust.M == 5) & (rfm_cust.recency > 142)]

kpis = {
    "gross_sales": float(sales.line_total.sum()),
    "tracked_customers": int(orders_per_customer.size),
    "repeat_rate": float((orders_per_customer >= 2).mean()),
    "yoy": float(jan_nov[2011] / jan_nov[2010] - 1),
    "new_cust_change": float(jun_nov_totals[2011] / jun_nov_totals[2010] - 1),
    "cancel_rate": float(-cancels.line_total.sum() / sales.line_total.sum()),
    "cancelled_value": float(-cancels.line_total.sum()),
    "guest_share": float(sales.loc[~sales.has_customer, "line_total"].sum() / sales.line_total.sum()),
    "top10_share": pareto[10][1] / 100,
    "champ_cust_share": float(seg.loc["Champions", "customer_share"]),
    "champ_rev_share": float(seg.loc["Champions", "revenue_share"]),
    "flagged_customers": int(len(flagged)),
    "flagged_rev": float(flagged.trailing_rev.sum()),
    "expected_loss": float(flagged.trailing_rev.sum() * precision),
    "rule_precision": precision,
    "top_tier_customers": int((flagged.value_quintile == 5).sum()),
    "top_tier_rev": float(flagged.loc[flagged.value_quintile == 5, "trailing_rev"].sum()),
    "quiet_vips": int(len(quiet)),
    "quiet_vips_rev": float(quiet.monetary.sum()),
}
pd.Series(kpis).to_frame("value")

,value
gross_sales,2.047626e+07
tracked_customers,5.878000e+03
repeat_rate,7.238857e-01
yoy,1.899986e-02
new_cust_change,-4.133077e-01
cancel_rate,7.142047e-02
cancelled_value,1.462424e+06
guest_share,1.514659e-01
top10_share,6.390000e-01
champ_cust_share,2.395066e-01


## 4. Chart tables

In [5]:
# Cohort heatmap rows: retention for months 0..12
cohort_rows = []
for c, grp in cohort.groupby("cohort"):
    by_k = grp.set_index("k").retention
    cohort_rows.append({
        "c": c.strftime("%Y-%m"),
        "n": int(grp.cohort_size.iloc[0]),
        "r": [None if (k not in by_k.index or pd.isna(by_k[k])) else round(float(by_k[k]), 4) for k in range(13)],
    })

repeat_rows = [
    {"g": "Dec 2009*" if g.startswith("Dec 2009") else g, "n": int(n), "rate": round(float(r), 4)}
    for g, n, r in zip(repeat90.group, repeat90.customers, repeat90.repeat_rate_90d)
]
order = ["Dec 2009*", "2010 H1", "2010 H2", "2011 H1", "2011 H2"]
repeat_rows.sort(key=lambda r: order.index(r["g"]))

new_rows = [
    {"m": r.cohort.strftime("%Y-%m"), "uk": int(r.UK), "non": int(r["Non-UK"])}
    for _, r in new_cust.iterrows()
    if pd.Timestamp("2010-01-01") <= r.cohort <= LAST_FULL   # Dec 2009 is left-censored, Dec 2011 partial
]

SLIPPING = {"Need Attention", "About to Sleep", "At Risk", "Can't Lose Them"}
segment_rows = records(rfm_seg.assign(slipping=rfm_seg.segment.isin(SLIPPING)).sort_values("revenue", ascending=False))

band_rows = [{"band": r.days_silent, "n": int(r.customers), "rate": round(float(r.churn_rate), 4)} for r in bands.itertuples()]

rule_rows = records(backtest[["rule", "flagged", "precision", "recall", "revenue_captured"]])

top_risk = (
    flagged[flagged.value_quintile == 5]
    .sort_values("trailing_rev", ascending=False)
    .head(12)
)
risk_rows = [
    {"id": int(r.customer_id), "days": int(r.recency), "avg_gap": None if pd.isna(r.avg_gap) else float(r.avg_gap),
     "orders": int(r.order_days), "trailing": float(r.trailing_rev), "early": bool(r.flag_early)}
    for r in top_risk.itertuples()
]

rec_rows = records(recs.rename(columns={"recommendation": "recommendation", "low_gbp": "low", "mid_gbp": "mid", "high_gbp": "high", "base_gbp": "base"})[
    ["rank", "recommendation", "evidence", "base", "low", "mid", "high", "effort"]])
print(len(cohort_rows), "cohorts |", len(segment_rows), "segments |", len(risk_rows), "priority customers |", len(rec_rows), "recommendations")

24 cohorts | 10 segments | 12 priority customers | 4 recommendations


## 5. Write the dashboard

In [6]:
payload = {
    "meta": {
        "period": "Dec 2009 to 9 Dec 2011",
        "lede": "Why growth is flat, who drives revenue, and who is about to leave. Built from 1.03M cleaned transactions of a UK online giftware retailer (UCI Online Retail II).",
        "footer": "Data: UCI Online Retail II, cleaned in notebook 01. Revenue is GBP. Churn = no purchase for 161 days after 30 Jun 2011. Scenario figures on the Actions tab are assumptions, not forecasts.",
    },
    "kpis": kpis,
    "monthly": records(monthly.round(2)),
    "cohorts": cohort_rows,
    "repeat90": repeat_rows,
    "newCust": new_rows,
    "segments": segment_rows,
    "pareto": pareto,
    "churnBands": band_rows,
    "rules": rule_rows,
    "topRisk": risk_rows,
    "recs": rec_rows,
}

# "<" is escaped so no text can close the script tag
data_json = json.dumps(payload, separators=(",", ":")).replace("<", "\\u003c")
template = (DASH_DIR / "template.html").read_text(encoding="utf-8")
assert template.count("__DATA__") == 1
out_path = DASH_DIR / "retail_dashboard.html"
out_path.write_text(template.replace("__DATA__", data_json), encoding="utf-8")
print(f"wrote {out_path.name}: {out_path.stat().st_size / 1e3:.0f} KB")

wrote retail_dashboard.html: 52 KB


## 6. Tableau-ready data files

One tidy CSV per chart family, with real date columns, long formats where a stacked chart needs them, and explicit sort keys. Connect each file to Tableau as its own text-file data source (no joins needed). See .

In [7]:
TABLEAU_DIR = DASH_DIR / "tableau_data"
TABLEAU_DIR.mkdir(exist_ok=True)

KPI_LABELS = {
    "gross_sales": ("Gross sales (GBP)", "currency"), "tracked_customers": ("Tracked customers", "count"),
    "repeat_rate": ("Repeat purchase rate", "percent"), "yoy": ("Sales growth, Jan-Nov 2011 vs 2010", "percent"),
    "new_cust_change": ("New customers, Jun-Nov 2011 vs 2010", "percent"), "cancel_rate": ("Cancelled value / gross sales", "percent"),
    "cancelled_value": ("Cancelled value (GBP)", "currency"), "guest_share": ("Sales with no customer ID", "percent"), "top10_share": ("Top 10% of customers' share of sales", "percent"),
    "champ_cust_share": ("Champions: share of customers", "percent"), "champ_rev_share": ("Champions: share of net revenue", "percent"),
    "flagged_customers": ("Customers silent 90+ days", "count"), "flagged_rev": ("Their trailing 12m sales (GBP)", "currency"),
    "expected_loss": ("Expected loss at rule precision (GBP)", "currency"), "rule_precision": ("90-day rule precision", "percent"),
    "top_tier_customers": ("Top-tier customers at risk", "count"), "top_tier_rev": ("Top-tier trailing 12m sales (GBP)", "currency"),
    "quiet_vips": ("Top-value customers silent 142+ days", "count"), "quiet_vips_rev": ("Their lifetime net sales (GBP)", "currency"),
}
tab_kpis = pd.DataFrame(
    [{"metric": k, "label": KPI_LABELS[k][0], "value": v, "format": KPI_LABELS[k][1]} for k, v in kpis.items()]
)

tab_monthly = monthly.assign(month=pd.to_datetime(monthly.m + "-01")).rename(columns={
    "gross": "gross_sales", "new": "new_customers", "returning": "returning_customers", "aov": "avg_order_value", "partial": "is_partial",
})[["month", "gross_sales", "cancelled", "orders", "customers", "new_customers", "returning_customers", "avg_order_value", "is_partial"]]

tab_new = (
    new_cust.melt(id_vars="cohort", value_vars=["UK", "Non-UK"], var_name="region", value_name="customers")
    .rename(columns={"cohort": "month"})
    .query("month >= '2010-01-01' and month <= @LAST_FULL")   # Dec 2009 is left-censored, Dec 2011 is partial
    .sort_values(["month", "region"])
)

tab_cohort = (
    cohort.dropna(subset=["retention"])                       # keep observable cells only
    .rename(columns={"cohort": "cohort_month", "k": "months_since_first_purchase"})
    .assign(active_customers=lambda d: (d.retention * d.cohort_size).round().astype(int), cohort_size=lambda d: d.cohort_size.astype(int))
    [["cohort_month", "months_since_first_purchase", "cohort_size", "active_customers", "retention"]]
)

tab_repeat = repeat90.assign(
    acquisition_period=lambda d: d.group.str.replace(" (left-censored)", "*", regex=False),
    sort_order=lambda d: d.group.map({"Dec 2009 (left-censored)": 1, "2010 H1": 2, "2010 H2": 3, "2011 H1": 4, "2011 H2": 5}),
).rename(columns={"customers": "customers", "repeat_rate_90d": "repeat_rate_90d"})[["acquisition_period", "sort_order", "customers", "repeat_rate_90d"]].sort_values("sort_order")

GROUPS = {"Champions": "Core", "Loyal": "Core", "Potential Loyalists": "Growing", "New Customers": "Growing",
          "Need Attention": "Slipping", "About to Sleep": "Slipping", "At Risk": "Slipping", "Can't Lose Them": "Slipping",
          "Hibernating": "Dormant", "Lost": "Dormant"}
tab_segments = rfm_seg.assign(segment_group=rfm_seg.segment.map(GROUPS)).sort_values("revenue", ascending=False)
tab_segments.insert(0, "revenue_rank", range(1, len(tab_segments) + 1))

tab_pareto = pd.DataFrame(pareto, columns=["pct_of_customers", "cum_pct_of_sales"])
tab_pareto["pct_of_customers"] = tab_pareto.pct_of_customers / 100
tab_pareto["cum_pct_of_sales"] = tab_pareto.cum_pct_of_sales / 100

tab_bands = bands.assign(sort_order=range(1, len(bands) + 1)).rename(columns={"customers": "customers", "median_rev": "median_trailing_sales"})

tab_rules = backtest.assign(
    rule_type=np.where(backtest.rule.str.contains("avg gap"), "Relative to own gap (3+ orders)", "Fixed days of silence"),
    sort_order=range(1, len(backtest) + 1),
)

tab_risk = (
    risk[risk.flag_90d & (risk.trailing_rev > 0)]
    .rename(columns={"recency": "days_silent", "avg_gap": "own_avg_gap_days", "trailing_rev": "trailing_12m_sales", "value_quintile": "value_tier", "flag_early": "early_flag"})
    [["customer_id", "days_silent", "order_days", "own_avg_gap_days", "trailing_12m_sales", "value_tier", "early_flag"]]
    .sort_values("trailing_12m_sales", ascending=False)
)

tab_recs = recs.rename(columns={"base_gbp": "measured_base_gbp"})[["rank", "recommendation", "evidence", "measured_base_gbp", "scenario_assumption", "low_gbp", "mid_gbp", "high_gbp", "effort", "confidence"]]

exports = {
    "kpis.csv": tab_kpis, "monthly.csv": tab_monthly, "new_customers_by_region.csv": tab_new, "cohort_retention.csv": tab_cohort,
    "repeat_rate_90d.csv": tab_repeat, "rfm_segments.csv": tab_segments, "revenue_concentration.csv": tab_pareto,
    "churn_by_days_silent.csv": tab_bands, "rule_backtest.csv": tab_rules, "top_at_risk_customers.csv": tab_risk, "recommendations.csv": tab_recs,
}
for name, frame in exports.items():
    frame.round(6).to_csv(TABLEAU_DIR / name, index=False)
print(f"exported {len(exports)} Tableau-ready CSVs to dashboard/tableau_data/")
pd.Series({n: len(f) for n, f in exports.items()}, name="rows").to_frame()

exported 11 Tableau-ready CSVs to dashboard/tableau_data/


C:\Users\mifta\AppData\Local\Temp\ipykernel_11880\2398093555.py:74: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  frame.round(6).to_csv(TABLEAU_DIR / name, index=False)
C:\Users\mifta\AppData\Local\Temp\ipykernel_11880\2398093555.py:74: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  frame.round(6).to_csv(TABLEAU_DIR / name, index=False)
C:\Users\mifta\AppData\Local\Temp\ipykernel_11880\2398093555.py:74: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  frame.round(6).to_csv(TABLEAU_DIR / name, index=False)


,rows
kpis.csv,19
monthly.csv,25
new_customers_by_region.csv,46
cohort_retention.csv,300
repeat_rate_90d.csv,5
rfm_segments.csv,10
revenue_concentration.csv,101
churn_by_days_silent.csv,6
rule_backtest.csv,10
top_at_risk_customers.csv,1365


### 6.1 Data for the generated Tableau workbook

`notebooks/09_build_tableau_workbook.ipynb` writes a ready-made workbook, `dashboard/tableau/retail_dashboard.twb`. To keep the workbook definition simple and reliable it reads a slightly different shape of data: one wide row for the headline tiles, stacked charts in long format, filters applied here instead of in Tableau (full months only, cohort months 1-12), and text tables pre-formatted as strings.

In [8]:
WB_DATA = DASH_DIR / "tableau" / "data"
WB_DATA.mkdir(parents=True, exist_ok=True)

k = lambda v: f"£{v / 1e3:,.0f}K"
slipping = tab_segments[tab_segments.segment_group == "Slipping"]

kpi_tiles = pd.DataFrame([{
    "gross_sales": kpis["gross_sales"], "tracked_customers": kpis["tracked_customers"], "repeat_rate": kpis["repeat_rate"],
    "sales_growth": kpis["yoy"], "new_customer_change": kpis["new_cust_change"],
    "slipping_customers": slipping.customers.sum(), "slipping_revenue": slipping.revenue.sum(), "slipping_revenue_share": slipping.revenue_share.sum(),
    "quiet_customers": kpis["quiet_vips"], "quiet_lifetime_sales": kpis["quiet_vips_rev"],
    "flagged_customers": kpis["flagged_customers"], "flagged_sales": kpis["flagged_rev"], "expected_loss": kpis["expected_loss"],
    "top_tier_customers": kpis["top_tier_customers"], "top_tier_sales": kpis["top_tier_rev"],
}])

# Tableau custom number formats cannot divide by a thousand or a million, so scaled copies are written here.
for col in ("gross_sales", "slipping_revenue"):
    kpi_tiles[col + "_m"] = kpi_tiles[col] / 1e6
for col in ("flagged_sales", "expected_loss", "top_tier_sales", "quiet_lifetime_sales"):
    kpi_tiles[col + "_k"] = kpi_tiles[col] / 1e3

full = tab_monthly[~tab_monthly.is_partial]
new_vs_returning = (
    full[["month", "returning_customers", "new_customers"]].melt(id_vars="month", var_name="customer_type", value_name="customers")
    # "Existing" sorts before "New", so Tableau's default colour order gives Existing the first (blue) colour
    .assign(customer_type=lambda d: d.customer_type.map({"returning_customers": "Existing", "new_customers": "New"}))
)

# Percent values are scaled to 0-100 in the data: Tableau axis number formats are not reliably applied.
seg_shares = (
    tab_segments.melt(id_vars=["revenue_rank", "segment"], value_vars=["customer_share", "revenue_share"], var_name="measure", value_name="share")
    .assign(measure=lambda d: d.measure.map({"customer_share": "Share of customers", "revenue_share": "Share of net revenue"}), share_pct=lambda d: d.share * 100)
    [["revenue_rank", "segment", "measure", "share_pct"]]
)

concentration = pd.DataFrame({"customers_pct": tab_pareto.pct_of_customers * 100, "sales_pct": tab_pareto.cum_pct_of_sales * 100})

# "2009 Dec*" sorts before "2010 H1" alphabetically, so Tableau orders the bars correctly without a sort setting
repeat_wb = tab_repeat.assign(
    acquisition_period=lambda d: d.acquisition_period.str.replace("Dec 2009*", "2009 Dec*", regex=False),
    repeat_pct=lambda d: d.repeat_rate_90d * 100,
)[["acquisition_period", "repeat_pct"]]

churn_wb = tab_bands.assign(churned_pct=lambda d: d.churn_rate * 100)[["sort_order", "days_silent", "churned_pct"]]

region_wb = tab_new.assign(region=lambda d: d.region.map({"UK": "Domestic (UK)", "Non-UK": "Overseas"}))

rule_table = pd.DataFrame({
    "Rank": tab_rules.sort_order, "Rule": tab_rules.rule, "Flagged": tab_rules.flagged.map("{:,}".format),
    "Precision": tab_rules.precision.map(lambda v: f"{v * 100:.0f}%"), "Recall": tab_rules.recall.map(lambda v: f"{v * 100:.0f}%"),
    "Churner revenue caught": tab_rules.revenue_captured.map(lambda v: f"{v * 100:.0f}%"),
})

top12 = tab_risk[tab_risk.value_tier == 5].head(12)
contact_list = pd.DataFrame({
    "Rank": range(1, len(top12) + 1), "Customer": "#" + top12.customer_id.astype(str), "Days silent": top12.days_silent.map("{:,}".format),
    "Own avg gap days": top12.own_avg_gap_days.map(lambda v: "n/a" if pd.isna(v) else f"{v:.0f}"), "Order days": top12.order_days.map("{:,}".format),
    "Trailing 12m sales": top12.trailing_12m_sales.map(lambda v: f"£{v:,.0f}"), "Early flag": top12.early_flag.map({True: "Yes", False: "No"}),
})

recs_table = pd.DataFrame({
    "Rank": tab_recs["rank"], "Action": tab_recs.recommendation, "Effort": tab_recs.effort, "Measured base": tab_recs.measured_base_gbp.map(k),
    "Low": tab_recs.low_gbp.map(k), "Mid": tab_recs.mid_gbp.map(k), "High": tab_recs.high_gbp.map(k),
})

# Tableau caches an extract per CSV file name; if a file's columns change, save it under a new name.
workbook_data = {
    "headline_kpis.csv": kpi_tiles,
    "monthly_gross_sales.csv": full[["month", "gross_sales"]].assign(gross_sales_m=lambda d: d.gross_sales / 1e6),
    "customers_existing_vs_new.csv": new_vs_returning,
    "new_customers_domestic_overseas.csv": region_wb,
    "cohort_heatmap.csv": tab_cohort[tab_cohort.months_since_first_purchase.between(1, 12)],
    "repeat_rate_90_days.csv": repeat_wb,
    "segment_shares_pct.csv": seg_shares,
    "revenue_concentration_pct.csv": concentration,
    "churn_by_silence.csv": churn_wb,
    "rule_table.csv": rule_table,
    "contact_list.csv": contact_list,
    "recommendations_table.csv": recs_table,
}
for name, frame in workbook_data.items():
    frame.round(6).to_csv(WB_DATA / name, index=False)
print(f"exported {len(workbook_data)} files to dashboard/tableau/data/")
pd.Series({n: len(f) for n, f in workbook_data.items()}, name="rows").to_frame()

exported 12 files to dashboard/tableau/data/


C:\Users\mifta\AppData\Local\Temp\ipykernel_11880\3866124938.py:82: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  frame.round(6).to_csv(WB_DATA / name, index=False)
C:\Users\mifta\AppData\Local\Temp\ipykernel_11880\3866124938.py:82: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  frame.round(6).to_csv(WB_DATA / name, index=False)
C:\Users\mifta\AppData\Local\Temp\ipykernel_11880\3866124938.py:82: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  frame.round(6).to_csv(WB_DATA / name, index=False)
C:\Users\mifta\AppData\Local\Temp\ipykernel_11880\3866124938.py:82: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  frame.round(6).to_csv(WB_DATA / name, index=False)


,rows
headline_kpis.csv,1
monthly_gross_sales.csv,24
customers_existing_vs_new.csv,48
new_customers_domestic_overseas.csv,46
cohort_heatmap.csv,210
repeat_rate_90_days.csv,5
segment_shares_pct.csv,20
revenue_concentration_pct.csv,101
churn_by_silence.csv,6
rule_table.csv,10


## 7. Open it

Open `dashboard/retail_dashboard.html` in a browser. Keyboard and screen-reader users can switch any chart to its **Table view**; the **Theme** button cycles auto, light and dark.